# 🚗 Road Lane Detection
### Pipeline: Grayscale → Gaussian Blur → Canny → ROI → Hough Lines


In [ ]:
# Install dependencies if needed
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Change this to your image or first frame path ──
IMAGE_PATH = 'your_image.jpg'   # e.g. from the Kaggle dataset


In [ ]:
# ── Load image ──────────────────────────────────────────
frame = cv2.imread(IMAGE_PATH)
assert frame is not None, f'Could not read {IMAGE_PATH}'
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,5))
plt.imshow(frame_rgb)
plt.title('Original')
plt.axis('off')
plt.show()


In [ ]:
# ── Step 1: Grayscale ────────────────────────────────────
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# ── Step 2: Gaussian Blur ────────────────────────────────
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# ── Step 3: Canny Edge Detection ─────────────────────────
edges = cv2.Canny(blurred, low_threshold=50, high_threshold=150)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, title in zip(axes,
                           [gray, blurred, edges],
                           ['Grayscale', 'Gaussian Blur', 'Canny Edges']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# ── Step 4: Region of Interest ───────────────────────────
height, width = frame.shape[:2]
mask = np.zeros_like(edges)

top_left     = (int(width * 0.42), int(height * 0.60))
top_right    = (int(width * 0.58), int(height * 0.60))
bottom_right = (int(width * 0.95), height)
bottom_left  = (int(width * 0.05), height)

polygon = np.array([[bottom_left, top_left, top_right, bottom_right]], dtype=np.int32)
cv2.fillPoly(mask, polygon, 255)
masked = cv2.bitwise_and(edges, mask)

# Visualise the ROI on the original
roi_viz = frame_rgb.copy()
cv2.polylines(roi_viz, polygon, isClosed=True, color=(255, 255, 0), thickness=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(roi_viz);   axes[0].set_title('ROI boundary');  axes[0].axis('off')
axes[1].imshow(masked, cmap='gray'); axes[1].set_title('Masked edges'); axes[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# ── Step 5: Hough Line Transform ─────────────────────────
lines = cv2.HoughLinesP(masked, rho=1, theta=np.pi/180,
                        threshold=40, minLineLength=60, maxLineGap=150)

def average_lines(frame, lines):
    left_fit, right_fit = [], []
    if lines is None:
        return None, None
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        if x2 == x1: continue
        slope = (y2 - y1) / (x2 - x1)
        intercept = y1 - slope * x1
        if abs(slope) < 0.3: continue
        (left_fit if slope < 0 else right_fit).append((slope, intercept))

    def make_coords(fit_list):
        if not fit_list: return None
        s, b = np.mean(fit_list, axis=0)
        y1 = frame.shape[0]
        y2 = int(y1 * 0.60)
        if s == 0: return None
        return (int((y1 - b) / s), y1, int((y2 - b) / s), y2)

    return make_coords(left_fit), make_coords(right_fit)

left_line, right_line = average_lines(frame, lines)

# Draw result
overlay = frame.copy()
for line, color in [(left_line, (255,50,50)), (right_line, (50,50,255))]:
    if line:
        x1,y1,x2,y2 = line
        cv2.line(overlay, (x1,y1), (x2,y2), color, 5)

if left_line and right_line:
    lx1,ly1,lx2,ly2 = left_line
    rx1,ry1,rx2,ry2 = right_line
    pts = np.array([[lx1,ly1],[lx2,ly2],[rx2,ry2],[rx1,ry1]])
    cv2.fillPoly(overlay, [pts], (0,255,0))

result = cv2.addWeighted(frame, 0.8, overlay, 0.4, 0)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,5))
plt.imshow(result_rgb)
plt.title('Lane Detection Result')
plt.axis('off')
plt.show()


In [ ]:
# ── Full Pipeline Summary (side-by-side) ─────────────────
stages = [
    (frame_rgb,   'Original'),
    (gray,        'Grayscale'),
    (blurred,     'Gaussian Blur'),
    (edges,       'Canny Edges'),
    (masked,      'ROI Masked'),
    (result_rgb,  'Final Result'),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (img, title) in zip(axes.flatten(), stages):
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
plt.suptitle('Road Lane Detection — Pipeline Overview', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('pipeline_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → pipeline_overview.png')


In [ ]:
# ── Batch process a folder ───────────────────────────────
INPUT_DIR  = 'dataset/'        # folder with your Kaggle images
OUTPUT_DIR = 'dataset_lanes/'  # results go here
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Import the helper from the script file if you placed it next to the notebook
# Otherwise the pipeline is fully self-contained above
print('Set INPUT_DIR to your Kaggle dataset path and run to batch process.')
